Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of large language models (LLMs) by giving them access to external,
up-to-date, and relevant information. Instead of relying solely on the knowledge encoded during their training,
RAG models can retrieve information from a separate knowledge base (like a database, documents, or the internet) and then use this retrieved context
to generate more accurate, relevant, and grounded responses.

In [ ]:
import os
os.environ['GROQ_API_KEY']='*YOUR GROQ API KEY*'


In [ ]:
from google.colab import files
upload= files.upload()

Saving drylab.pdf to drylab (2).pdf


In [ ]:
pip uninstall -y langchain langchain-community openai

Found existing installation: langchain 1.3.4
Uninstalling langchain-1.3.4:
  Successfully uninstalled langchain-1.3.4
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: openai 2.41.0
Uninstalling openai-2.41.0:
  Successfully uninstalled openai-2.41.0


In [ ]:
pip install -q langchain langchain_community langchain_openai faiss-cpu pypdf sentence-transformers

In [ ]:
pip install -q langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.8 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import  PyPDFLoader
pdf_file= list(upload.keys())[0]
print(pdf_file)
loader= PyPDFLoader(pdf_file)
documents = loader.load()
print("Total pages loaded:",len(documents))

drylab (2).pdf
Total pages loaded: 3


In [ ]:
print(documents[2].page_content)

Paralinx about camera integration; Amazon,
Google and IBM about cloud computing.
WWDC and Silicon Valley: We were very
pleasantly surprised to be invited by Apple to
their World Wide Developers Conference in
San Jose in June, despite not having applied.
It's a valuable chance to learn and make new
connections. We’re also setting aside time to
meet other potential partners.
Cine Gear: We have decided not to attend
the Cine Gear expo in L.A. this year, since
feedback from many users about the show
were mixed, and our planned beta version of
3.0 is slightly delayed.
Development and launch: Development
is around one month behind our original
schedule. We expect the delay to decrease,
with new developers on board.
The launch of Drylab 3.0 will take place at
the International Broadcasters Convention
in Amsterdam in September, and we are
working hard to get solid feedback from pilot
users before then.
Annual General Meeting: Drylab's AGM
will be held on June 16th at 15:00. An
invitation will 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter= RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs= text_splitter.split_documents(documents)
print("Total chunks:",len(docs))

Total chunks: 8


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings= HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstores= FAISS.from_documents(docs,embeddings)
print("Vector DB successfully created")
print(vectorstores)
vectorstores.save_local("faiss_index")

Vector DB successfully created


In [ ]:
retriever = vectorstores.as_retriever(search_kwargs={'k':3})
print(retriever)

tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7de3ed5b7470> search_kwargs={'k': 3}


In [ ]:
print(vectorstores.index_to_docstore_id)

{0: '4b14a351-bacf-438c-9d89-e8c237c07fce', 1: 'c7e5d9a3-9759-428e-8b43-c85e1fde1c17', 2: '005de127-0906-459a-8d52-4006a8a5d627', 3: 'bbca68f2-9509-43fb-b71d-98a1c1504cd5', 4: '98068502-7806-45e4-9d33-23d510731056', 5: 'b9df0e5c-38ff-495d-a81b-99e48c984835', 6: '7f644178-b9f0-4176-819f-f2fa291c8597', 7: 'ca34cc58-8311-4075-a161-2d1f6da5b448'}


In [ ]:
docs_list= list(vectorstores.docstore._dict.values())
print(docs_list[0].page_content)

DrylabNewsfor investors & friends · May 2 017
Welcome to our first newsletter of 2017! It's
been a while since the last one, and a lot has
happened. We promise to keep them coming
every two months hereafter, and permit
ourselves to make this one rather long. The
big news is the beginnings of our launch in
the American market, but there are also
interesting updates on sales, development,
mentors and (of course) the investment
round that closed in January.
New capital: The investment round was
successful. We raised 2.13 MNOK to match
the 2.05 MNOK loan from Innovation
Norway. Including the development
agreement with Filmlance International, the
total new capital is 5 MNOK, partly tied to
the successful completion of milestones. All
formalities associated with this process are
now finalized.
New owners: We would especially like to
warmly welcome our new owners to the
Drylab family: Unni Jacobsen, Torstein Jahr,
Suzanne Bolstad, Eivind Bergene, Turid Brun,


In [ ]:
for i in range(vectorstores.index.ntotal):
    vector = vectorstores.index.reconstruct(i)
    print(f"vector {i}: ")
    print(vector)
    print("-"*50)

vector 0: 
[-3.91199142e-02 -5.94291836e-02  1.46298939e-02 -1.62920617e-02
  5.23726605e-02 -5.83062395e-02 -2.93215569e-02  6.08805940e-02
 -1.77485645e-02  3.70959863e-02 -9.63691715e-03 -5.82237504e-02
 -9.79659241e-03 -2.49545239e-02  5.61320782e-03 -2.94187721e-02
 -4.14771326e-02 -3.69341150e-02 -3.54975201e-02  3.47201489e-02
  2.65763514e-02 -5.86766303e-02  6.55697063e-02 -4.41529155e-02
  1.50383532e-01  1.08384490e-02  7.22287744e-02  7.79628847e-03
 -2.33278219e-02 -9.20308232e-02 -1.31974882e-02  5.53362630e-02
  2.09714174e-02 -3.84281874e-02  1.16390564e-01  1.02809235e-01
  2.35380810e-02 -1.48531944e-02 -8.02012067e-03  2.43354477e-02
 -3.06907413e-03 -4.77203466e-02  4.52364720e-02 -4.22468781e-02
  6.53807353e-03 -2.79499847e-03 -1.44934440e-02  9.63014271e-03
 -2.31962744e-02  9.93745923e-02 -2.95315031e-02 -1.06358126e-01
  4.94883545e-02 -7.07823550e-03 -8.07749778e-02  2.98487972e-02
 -7.28611872e-02  2.69991788e-03 -3.53085506e-03 -8.98272078e-03
  7.00865462e-

In [ ]:
from langchain_openai import ChatOpenAI
import os
llm= ChatOpenAI(model="llama-3.3-70b-versatile",
                api_key=os.getenv('GROQ_API_KEY'),
                base_url="http://api.groq.com/openai/v1")

In [ ]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY')
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_template(
    """ Answer the question base only on the context below:
    {context}
    Question: {question}
    """
)


# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG pipeline (modern way)
rag_chain = (
    {"context": retriever | format_docs, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)


query = input("Ask a question: ")

result = rag_chain.invoke(query)

print("\n Answer:\n")
print(result)

Ask a question: Sales

 Answer:

The sales information mentioned in the context is as follows:

- The return customer rate is now 80%, proving value and willingness to pay.
- Film Factory Montreal is the first customer in Canada.
- Lumiere Numeriques have started using the service in France.
- There are new customers in Norway, including high-profile users such as Gareth Unwin, producer of the Oscar-winning film "The King's Speech".
- The revenue for the first four months is 200 kNOK, compared to 339 kNOK for all of 2016.
- A partnership is being worked on to safeguard sales in Norway while focusing more on the US.


embedding modles,compare,diff chunks,multi pdf,reranking